In [73]:
import pandas as pd
import re
import ast
from html import escape
from IPython.display import display, HTML

In [74]:
df = pd.read_csv(r'C:\Users\joly-\Github\HUMAN\news\final_df\combined_final_df.csv', index_col=0)
df.head()

,title,outlet,date,authors,body,word_count,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,...,verbs,date_parsed,year,topic,probability,topic_norm,topic_label_nl,topic_meta_nl,topic_label,topic_meta
0,Chatbot Emma glimlacht na een compliment Chatb...,Parool,2015-10-23,BART VAN ZOELEN,"'Dag Denise, hoe oud ben jij?"" vragen we Denis...",238,yes,['chatbot'],['chatbots'],"['chatbot', 'chatbots']",...,vragen praten antwoorden vat gaan proberen kij...,2015-10-23,2015,8,0.153558,8,facebook en algoritmen,SOCIALE MEDIA,facebook and algorithms,SOCIAL MEDIA
1,Vera en Roos hebben het druk Vera en Roos hebb...,Parool,2015-10-23,BART VAN ZOELEN,Klantenservice: Virtuele medewerkers kunnen st...,249,yes,[],"['chatbot', 'chatbots']","['chatbot', 'chatbots']",...,beantwoorden beantwoorden krijgen doen zitten ...,2015-10-23,2015,8,0.212454,8,facebook en algoritmen,SOCIALE MEDIA,facebook and algorithms,SOCIAL MEDIA
2,Leer van big data-incidenten Leer van big data...,Parool,2015-01-03,Unknown Authors,Keerpunt: De doorbraak van big data \nBracht 2...,303,yes,[],['slimme algoritmes'],['slimme algoritmes'],...,brengen gaan brengen beleven overgeven hechten...,2015-01-03,2015,12,0.128177,12,bitcoin,BEDRIJF & FINANCIËN,bitcoin,BUSINESS & FINANCE
3,Brieven Brieven,Parool,2015-09-03,Unknown Authors,Geen vrije keuze \nVrijdag had Het Parool een ...,310,yes,[],['algoritme'],['algoritme'],...,wassen zeggen gaan afspreken komen ontstaan de...,2015-09-03,2015,14,0.373948,14,onderwijs,ONDERWIJS,eductation,EDUCATION
4,Stad van stenen en pixels Stad van stenen en p...,Parool,2015-05-19,TRACY METZ,"De talkshow Stadsleven, deze maand tegelijk me...",317,yes,[],['algoritme'],['algoritme'],...,gaan vertrouwen leven fietsen bestaan loop fie...,2015-05-19,2015,8,1.000000,8,facebook en algoritmen,SOCIALE MEDIA,facebook and algorithms,SOCIAL MEDIA


In [75]:
def inspect_ai_related(df, body='body', num_rows=30, keyword_col='matched_keywords_all'):
    """
    Displays rows with highlighted keywords.
    Always highlights matched_keywords_all (AI keywords); also highlights keyword_col if different.
    """

    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if body not in df.columns:
        raise KeyError(f"Missing required column: {body}")
    if keyword_col not in df.columns:
        raise KeyError(f"Missing keyword column: {keyword_col}")

    n = min(num_rows, len(df))
    samples = df.head(n)

    def parse_col(val):
        if isinstance(val, str):
            try:
                val = ast.literal_eval(val)
            except (ValueError, SyntaxError):
                return set()
        return {str(x).lower() for x in val} if isinstance(val, (list, set, tuple)) else set()

    def highlight_keywords(text, keywords):
        if pd.isna(text):
            return ""
        s = str(text)
        kws = {str(w).strip().lower() for w in keywords if str(w).strip()}
        if not kws:
            return s
        ordered = sorted(kws, key=len, reverse=True)
        def make_pattern(word):
            return rf'(?<![A-Za-z0-9]){re.escape(word)}(?![A-Za-z0-9])'
        combined = "|".join(make_pattern(w) for w in ordered)
        regex = re.compile(combined, flags=re.IGNORECASE)
        def repl(m):
            return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{escape(m.group(0))}</span>'
        return regex.sub(repl, s)

    for idx, row in samples.iterrows():
        ai_val    = row['ai_related']
        title     = row['title']
        body_text = row[body]

        ai_terms    = parse_col(row.get('matched_keywords_all', []))
        extra_terms = parse_col(row.get(keyword_col, [])) if keyword_col != 'matched_keywords_all' else set()
        row_terms   = ai_terms | extra_terms

        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body_text, row_terms)

        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if ai_terms:
            header_html += f'<br><strong>ai keywords:</strong> {sorted(ai_terms)}'
        if extra_terms:
            header_html += f'<br><strong>{keyword_col}:</strong> {sorted(extra_terms)}'
        header_html += '</div>'

        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} articles.")
    return samples

In [76]:
def to_list(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

# df['matched_keywords_all'] = df.apply(
#     lambda row: to_list(row['company_hits']) + to_list(row['matched_keywords_all']),
#     axis=1
# )




In [77]:
noise_keywords = ["weiwei", "wei wei", "aiaiai"]

_NOISE_PAT = re.compile(
      r"(ai\s*,\s*ai\s*,\s*ai|" + "|".join(re.escape(k) for k in noise_keywords) + r")",
      re.IGNORECASE
  )

In [78]:
# noise_keywords = [k for k in "ai, ai, ai|Ai, ai, ai|AIAIAI|wei wei|Wei Wei|WeiWei|weiwei|aiaiai".split('|') if k.strip()]

# _NOISE_PAT = re.compile(
#     r"(" + "|".join(re.escape(k) for k in noise_keywords) + r")",
#     re.IGNORECASE
# )

def find_noise_words(row):
    title = str(row['title']) if pd.notna(row['title']) else ""
    body  = str(row['body'])  if pd.notna(row['body'])  else ""
    hits  = _NOISE_PAT.findall(title) + _NOISE_PAT.findall(body)
    return sorted({h.lower() for h in hits})

df['ai_words'] = df.apply(find_noise_words, axis=1)
df[df['ai_words'].map(len) > 0][['title', 'body', 'ai_words']].head(10)

,title,body,ai_words
26,Argument Argument,"WiBra \nDe laatste paar dagen, sinds de jurk v...",[aiaiai]
35,Ai Wei Wei Ai Wei Wei,"Tv-tip \nAi Wei Wei: never sorry \nZondag, Can...",[wei wei]
254,Ai Peking Ai; Peking,"Doffe ogen, een gezicht getekend door het leve...",[wei wei]
280,Xi Jinping heeft van Engeland geen last Xi Ji...,Joost van Mierlo Londen Van mensenrechten to...,[wei wei]
291,'Al schaatsend heb ik overwogen om de Elfstede...,Zijn rechtvaardigheidsgevoel is een van de gro...,"[ai, ai, \nai]"
386,Kan Duitslands knuffelcabaretier nog wel prikk...,Analyse Jan Böhmermann \nVanavond keert hij te...,[wei wei]
602,'De basis onder mijn voeten is twee keer wegge...,interview Ebru Umar wil graag weer haar werk k...,[wei wei]
1167,Opgevist Opgevist,KUNST Sarah van Sonsbeeck\nSarah van Sonsbeeck...,[wei wei]
2843,MEESTERKNECHT MET TWIJFELS MEESTERKNECHT MET T...,Antwan Tolhoek wacht zware taak als rechterhan...,"[ai, ai, ai]"
3170,China heeft het Westen niet langer rnodig Chin...,VPRO’s Tegenlicht wijdde af­gelopen zondag een...,[wei wei]


In [79]:
# Filter rows where ai_words is a non-empty list
noise_articles = df[df['ai_words'].apply(lambda x: isinstance(x, list) and len(x) > 0)]

In [80]:
print(len(noise_articles), "articles found with noise keywords.")

23 articles found with noise keywords.


In [81]:
noise_articles['ai_words'].iloc[0]

['aiaiai']

In [82]:
# program = "all" to display from all programs, or specify a program like "jinek"
inspect_ai_related(noise_articles, body = 'body', num_rows=23, keyword_col='ai_words')

Displayed 23 articles.


,title,outlet,date,authors,body,word_count,ai_related,matched_keywords_title,matched_keywords_body,matched_keywords_all,...,date_parsed,year,topic,probability,topic_norm,topic_label_nl,topic_meta_nl,topic_label,topic_meta,ai_words
26,Argument Argument,VK,2015-05-13,AAF BRANDT CORSTIUS,"WiBra \nDe laatste paar dagen, sinds de jurk v...",475,yes,[],['ai'],['ai'],...,2015-05-13,2015,1,0.456148,1,muziek,"MEDIA, KUNST, CULTUUR & SPORT",music,"MEDIA, ARTS, CULTURE & SPORTS",[aiaiai]
35,Ai Wei Wei Ai Wei Wei,VK,2015-09-26,Unknown Authors,"Tv-tip \nAi Wei Wei: never sorry \nZondag, Can...",65,yes,['ai'],['ai'],['ai'],...,2015-09-26,2015,3,0.151157,3,taal,MAATSCHAPPIJ,language,SOCIETY,[wei wei]
254,Ai Peking Ai; Peking,TG,2015-03-26,"Vink, Marcel","Doffe ogen, een gezicht getekend door het leve...",311,yes,['ai'],['ai'],['ai'],...,2015-03-26,2015,3,0.169630,3,taal,MAATSCHAPPIJ,language,SOCIETY,[wei wei]
280,Xi Jinping heeft van Engeland geen last Xi Ji...,TG,2015-10-23,Unknown Authors,Joost van Mierlo Londen Van mensenrechten to...,340,yes,[],['ai'],['ai'],...,2015-10-23,2015,3,0.282406,3,taal,MAATSCHAPPIJ,language,SOCIETY,[wei wei]
291,'Al schaatsend heb ik overwogen om de Elfstede...,TG,2015-01-03,"Stavrianos, Katina",Zijn rechtvaardigheidsgevoel is een van de gro...,1309,yes,[],['ai'],['ai'],...,2015-01-03,2015,3,0.075175,3,taal,MAATSCHAPPIJ,language,SOCIETY,"[ai, ai, \nai]"
386,Kan Duitslands knuffelcabaretier nog wel prikk...,VK,2016-05-12,STERRE LINDHOUT,Analyse Jan Böhmermann \nVanavond keert hij te...,1123,yes,[],['ai'],['ai'],...,2016-05-12,2016,1,0.030629,1,democratie,POLITIEK & RECHT,democracy,POLITICS & LAW,[wei wei]
602,'De basis onder mijn voeten is twee keer wegge...,TR,2016-05-04,MAAIKE BEZEMER,interview Ebru Umar wil graag weer haar werk k...,551,yes,[],['ai'],['ai'],...,2016-05-04,2016,1,0.052398,1,democratie,POLITIEK & RECHT,democracy,POLITICS & LAW,[wei wei]
1167,Opgevist Opgevist,GA,2017-05-25 00:00:00,Unknown Authors,KUNST Sarah van Sonsbeeck\nSarah van Sonsbeeck...,555,yes,[],['ai'],['ai'],...,2017-05-25,2017,8,0.122892,8,kunst,"MEDIA, KUNST, CULTUUR & SPORT",art,"MEDIA, ARTS, CULTURE & SPORTS",[wei wei]
2843,MEESTERKNECHT MET TWIJFELS MEESTERKNECHT MET T...,TG,2019-05-23,Léon de Kort,Antwan Tolhoek wacht zware taak als rechterhan...,542,yes,[],['ai'],['ai'],...,2019-05-23,2019,6,0.057008,6,schrijven en cabaret,"MEDIA, KUNST, CULTUUR & SPORT",writing and cabaret,"MEDIA, ARTS, CULTURE & SPORTS","[ai, ai, ai]"
3170,China heeft het Westen niet langer rnodig Chin...,VK,2020-06-02,Heleen Mees,VPRO’s Tegenlicht wijdde af­gelopen zondag een...,682,yes,[],['ai'],['ai'],...,2020-06-02,2020,2,0.262592,2,coronapandemie,MAATSCHAPPIJ,corona pandemic,SOCIETY,[wei wei]


In [83]:
# if 'ai' is the only keyword and ai_words is not [], delete rows from df
def is_noise(row):
    ai_terms = to_list(row.get('matched_keywords_all', []))
    noise_terms = to_list(row.get('ai_words', []))
    return set(ai_terms) == {'ai'} and len(noise_terms) > 0
noise_mask = df.apply(is_noise, axis=1)
print(f"Removing {noise_mask.sum()} noisy articles.")
df_cleaned = df[~noise_mask].copy()
print(f"Cleaned DataFrame has {len(df_cleaned)} articles (removed {noise_mask.sum()}).")


Removing 22 noisy articles.
Cleaned DataFrame has 13209 articles (removed 22).


In [84]:
# save cleaned df to csv
df_cleaned.to_csv(r'C:\Users\joly-\Github\HUMAN\news\final_df\cleaned_final_df.csv', index=False)